# 006 - Transformer Deep Dive

Comprehensive guide to all transformer categories:
1. **Normalizers** — whitespace, nulls, case, enums, timestamps
2. **Field Transformers** — JSON flatten, UUID gen, regex clean, timestamp normalize
3. **Document Transformers** — JSON→rows, text chunker, PII redactor
4. **Filter Transformers** — row filter, column pruner
5. **Graph Transformers** — row→node, infer edges
6. **Truncators** — length, field count, token
7. **Custom Transformers** — how to write your own


In [ ]:
!pip install -q --force-reinstall --no-deps ~/SageMaker/document-graph/wheels/document_graph-3.0.3-py3-none-any.whl


In [ ]:
import json
from document_graph.transform.transformer_provider_config import TransformerProviderConfig
print('Imports OK')


In [ ]:
# Sample data for all examples
records = [
    {'id': '1', 'name': '  Alice Smith  ', 'email': ' alice@corp.com ', 'role': 'admin', 'metadata': '{"region": "us-east-1", "tier": "prod"}', 'notes': ''},
    {'id': '2', 'name': 'Bob Jones', 'email': 'bob@corp.com', 'role': 'viewer', 'metadata': '{"region": "eu-west-1", "tier": "dev"}', 'notes': 'N/A'},
    {'id': '3', 'name': 'Charlie Brown', 'email': 'charlie@corp.com', 'role': 'ADMIN', 'metadata': '{"region": "us-west-2", "tier": "staging"}', 'notes': None},
]
print(f'Sample: {len(records)} records')


## 1. Normalizers


In [ ]:
from document_graph.transform.normalizers.normalize_whitespace_provider import NormalizeWhitespaceProvider
from document_graph.transform.normalizers.normalize_nulls_provider import NormalizeNullsProvider
from document_graph.transform.normalizers.normalize_case_provider import NormalizeCaseProvider

# Whitespace
result = NormalizeWhitespaceProvider(TransformerProviderConfig(name='ws', args={})).transform(records)
print(f'Whitespace: "{records[0]["name"]}" → "{result[0]["name"]}"')

# Nulls
result = NormalizeNullsProvider(TransformerProviderConfig(name='nulls', args={})).transform(records)
print(f'Nulls: notes={[r["notes"] for r in result]}')

# Case
result = NormalizeCaseProvider(TransformerProviderConfig(name='case', args={'fields': ['role'], 'case': 'lower'})).transform(records)
print(f'Case: roles={[r["role"] for r in result]}')


## 2. Field Transformers


In [ ]:
from document_graph.transform.field_transformers.json_flattener import JSONFlattenerProvider
from document_graph.transform.field_transformers.uuid_generator import UuidGeneratorTransformer

# JSON Flatten (prefix defaults to field name, no separator)
result = JSONFlattenerProvider(TransformerProviderConfig(name="flat", args={"field": "metadata"})).transform(records)
flat_keys = [k for k in result[0].keys() if k.startswith("metadata")]
print(f"JSON flatten added keys: {flat_keys}")

# UUID Generator (arg is target_field)
result = UuidGeneratorTransformer(TransformerProviderConfig(name="uuid", args={"target_field": "graph_id"})).transform(records)
uid = result[0]["graph_id"]
print(f"UUID: {uid}")


## 3. Document Transformers


In [ ]:
from document_graph.transform.document_transformers.json_to_rows import JSONToRowsTransformer
from document_graph.transform.document_transformers.text_chunker import TextChunkerTransformer

# JSON to Rows
json_records = [{'data': '[{"x":1},{"x":2},{"x":3}]'}]
result = JSONToRowsTransformer(TransformerProviderConfig(name='j2r', args={'field': 'data'})).transform(json_records)
print(f'JSON→Rows: {len(json_records)} record → {len(result)} rows')

# Text Chunker
text_records = [{'content': 'A' * 500}]
result = TextChunkerTransformer(TransformerProviderConfig(name='chunk', args={'field': 'content', 'chunk_size': 100})).transform(text_records)
print(f'Chunker: 500 chars → {len(result)} chunks of ~100')


## 4. Filter Transformers


In [ ]:
from document_graph.transform.filter_transformers.row_filter import RowFilterProvider
from document_graph.transform.filter_transformers.column_pruner import ColumnPrunerProvider

# Row Filter
result = RowFilterProvider(TransformerProviderConfig(name='rf', args={'field': 'role', 'operator': 'ne', 'value': 'viewer'})).transform(records)
print(f'Row filter (exclude viewers): {len(records)} → {len(result)}')

# Column Pruner
result = ColumnPrunerProvider(TransformerProviderConfig(name='cp', args={'keep': ['id', 'name', 'role']})).transform(records)
print(f'Column prune: {list(result[0].keys())}')


## 5. Graph Transformers


In [ ]:
from document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer
from document_graph.transform.graph_transformers.infer_edges import EdgeInferencer

# Row → Node
nodes = RowToNodeTransformer(TransformerProviderConfig(name='r2n', args={'type': 'User'})).transform(records)
print(f'Nodes: {len(nodes)} (type={nodes[0].get("node_type", "User")})')

# Infer Edges (shared field = relationship)
edge_records = [{'id': '1', 'account': 'A'}, {'id': '2', 'account': 'A'}, {'id': '3', 'account': 'B'}]
edges = EdgeInferencer(TransformerProviderConfig(name='edges', args={'source_field': 'account', 'edge_type': 'SAME_ACCOUNT'})).transform(edge_records)
print(f'Edges: {len(edges)} inferred from shared account field')


## 6. Truncators


In [ ]:
from document_graph.transform.truncators.length_truncator import LengthTruncator

long_records = [{"id": "1", "bio": "x" * 1000}]
result = LengthTruncator(TransformerProviderConfig(name="trunc", args={"field": "bio", "max_length": 100})).transform(long_records)
bio_len = len(result[0]["bio"])
print(f"Truncator: 1000 chars → {bio_len} chars")


## 7. Writing a Custom Transformer


In [ ]:
from document_graph.transform.transformer_provider_base import TransformerProvider

class UpperCaseTransformer(TransformerProvider):
    """Custom transformer: uppercase specified fields."""
    def transform(self, records):
        fields = self.config.args.get('fields', [])
        return [{**r, **{f: r[f].upper() for f in fields if f in r and isinstance(r[f], str)}} for r in records]

config = TransformerProviderConfig(name='upper', args={'fields': ['name', 'email']})
result = UpperCaseTransformer(config).transform(records)
print(f'Custom: {result[0]["name"]}, {result[0]["email"]}')


## Summary

| Category | Transformers | Purpose |
|----------|-------------|----------|
| Normalizers | whitespace, nulls, case, enum, timestamp, spelling | Clean & standardize |
| Field | json_flattener, uuid_gen, regex_clean, comma_split, timestamp | Reshape fields |
| Document | json_to_rows, text_chunker, pii_redactor | Split/transform docs |
| Filter | row_filter, column_pruner | Remove unwanted data |
| Graph | row_to_node, infer_edges | Create graph structures |
| Truncators | length, field_count, token | Limit data size |

All transformers follow the same interface: `TransformerProvider(config).transform(records) → records`
